In [1]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')

import pandas as pd

eval_table = pd.read_csv('data/processed/final_evaluation_table.csv')
print(eval_table.shape)
print(eval_table.columns.tolist())

(20, 22)
['StockCode', 'Prophet_MAE', 'Prophet_RMSE', 'Prophet_MAPE', 'Naive_MAE', 'Naive_RMSE', 'Naive_MAPE', 'SeasonalNaive_MAE', 'SeasonalNaive_RMSE', 'SeasonalNaive_MAPE', 'BeatNaive', 'BeatSeasonalNaive', 'DataQualityFlag', 'MeanRelativeUncertainty', 'MeanForecast', 'TrendDirection', 'TrendChangeUnits', 'TrendChangeVsMean%', 'RiskStatus', 'Description', 'ImprovementOverNaive%', 'ImprovementOverSeasonal%']


In [2]:
reliable = eval_table[eval_table['DataQualityFlag'] == 'OK'].sort_values('Prophet_MAE')

print(reliable[['StockCode', 'Description', 'Prophet_MAE', 'Naive_MAE', 
                 'SeasonalNaive_MAE', 'BeatNaive', 'BeatSeasonalNaive',
                 'ImprovementOverNaive%', 'ImprovementOverSeasonal%',
                 'MeanRelativeUncertainty', 'TrendDirection', 'RiskStatus']].to_string(index=False))

StockCode                        Description  Prophet_MAE  Naive_MAE  SeasonalNaive_MAE  BeatNaive  BeatSeasonalNaive  ImprovementOverNaive%  ImprovementOverSeasonal%  MeanRelativeUncertainty TrendDirection RiskStatus
    22386            JUMBO BAG PINK POLKADOT       143.87     107.62             256.00      False               True                  -33.7                      43.8                    270.8        Growing  Low Stock
    21212    PACK OF 72 RETROSPOT CAKE CASES       161.28     213.12             402.12       True               True                   24.3                      59.9                    217.2      Declining   Adequate
    84946       ANTIQUE SILVER T-LIGHT GLASS       171.00     171.00             195.62      False               True                    0.0                      12.6                    210.2        Growing  Low Stock
   85099F               JUMBO BAG STRAWBERRY       176.87     140.25             171.25      False              False           

In [3]:
print("Win rate:")
print(f"  Beats naive: {reliable['BeatNaive'].sum()}/{len(reliable)}")
print(f"  Beats seasonal naive: {reliable['BeatSeasonalNaive'].sum()}/{len(reliable)}")

print("\nAverage MAE:")
print(f"  Prophet: {reliable['Prophet_MAE'].mean():.1f}")
print(f"  Naive: {reliable['Naive_MAE'].mean():.1f}")
print(f"  Seasonal Naive: {reliable['SeasonalNaive_MAE'].mean():.1f}")

print("\nImprovement over naive - mean: {:.1f}% | median: {:.1f}%".format(
    reliable['ImprovementOverNaive%'].mean(), reliable['ImprovementOverNaive%'].median()
))
print("Improvement over seasonal naive - mean: {:.1f}% | median: {:.1f}%".format(
    reliable['ImprovementOverSeasonal%'].mean(), reliable['ImprovementOverSeasonal%'].median()
))

Win rate:
  Beats naive: 7/18
  Beats seasonal naive: 13/18

Average MAE:
  Prophet: 336.1
  Naive: 379.5
  Seasonal Naive: 546.4

Improvement over naive - mean: -13.1% | median: -3.2%
Improvement over seasonal naive - mean: -1.6% | median: 27.6%


## Day 5 — Notes & Observations

**Data loading cell** — loaded `final_evaluation_table.csv`, which combines the outputs of all previous analyses (baseline comparison, uncertainty metrics, trend analysis, and inventory risk) into a single table containing 20 products and 22 evaluation columns.

**Full evaluation table** — displayed all 18 products flagged as `OK`, sorted by Prophet MAE, allowing direct comparison of forecasting accuracy, uncertainty, trend strength, and inventory status in one view.

**Summary statistics cell** — calculated overall win rates, average MAE values, and mean/median improvement metrics directly from the merged table.

---

## Fix applied earlier today: MAPE near-zero inflation

`calculate_mape` originally excluded only weeks where actual demand was exactly zero. However, weeks with very small but non-zero demand (for example, 1–2 units) still produced extremely large percentage errors.

The function was updated to exclude weeks where actual demand was below 5 units.

As a result:

- Prophet's average MAPE (OK products only) dropped from **292.2%** to **122.9%**
- Product **21915** dropped from **3517%** to **470.2%**

This was not a modeling improvement—it was a correction to make the metric itself more meaningful.

---

## Final evaluation (18 OK products)

### Win rates

- Prophet beats the naive baseline on **7 of 18 products (39%)**
- Prophet beats the seasonal naive baseline on **13 of 18 products (72%)**

### Average MAE

- Prophet: **336.1**
- Naive: **379.5**
- Seasonal naive: **546.4**

Prophet achieves the lowest average absolute error overall.

### Improvement over naive baseline

- Mean improvement: **−13.1%**
- Median improvement: **−3.2%**

These numbers suggest that the typical product slightly favors the naive baseline. Prophet's advantage comes primarily from large wins on a smaller number of products rather than consistent improvements across all products.

### Improvement over seasonal naive baseline

- Median improvement: **+27.6%**

The median product benefits substantially from Prophet compared with seasonal naive. However, the average improvement is much smaller because a few products experience very large losses, which pull the mean downward.

This difference between mean and median is itself an important finding: Prophet improves forecasting performance for most products, but the gains are not evenly distributed.

---

## Key conclusion

Prophet's trend-only model does not consistently outperform a simple "repeat last week" strategy. Although Prophet achieves the lowest average MAE, naive forecasting performs slightly better on the typical product.

Against seasonal naive, however, Prophet shows a clear advantage, outperforming it on nearly three-quarters of products and reducing error by **27.6%** for the median product.

The main lesson is that average metrics alone can be misleading. Both win rates and mean/median comparisons are needed to understand model performance properly.

---

## Products flagged for Week 5 investigation

### Product 21977

Prophet significantly overestimated demand during the test period, forecasting approximately **385–391 units per week** while actual demand remained much lower.

This appears to be caused by trend overshoot: the product showed a growing historical trend, and without seasonal components the model continued projecting that growth into a period where demand flattened.

No changes were made in Week 4 because tuning the model against a single holdout window would risk overfitting.

### Product 15036

Product 15036 showed the worst relative performance against seasonal naive, with an improvement score of **−312.3%**.

At this stage, it is impossible to determine whether this reflects a genuine product-specific pattern or simply an artifact of the single test split. Week 5's walk-forward validation will determine whether this underperformance is consistent across multiple time windows.

---

## Decision for Week 5

Prophet remains the primary forecasting model.

Although the naive baseline performs slightly better on many stable products, Prophet still achieves the lowest average MAE, clearly outperforms the seasonal naive baseline, and provides additional outputs—trend estimates, uncertainty intervals, and inventory insights—that are essential for downstream business tasks.

Week 4 relies on a single train/test split, so replacing Prophet with a simpler baseline at this stage would be premature. Week 5's walk-forward validation will determine whether Prophet's strengths and weaknesses remain consistent across multiple time periods.